# MLOps Assignment 01 - Heart Disease Risk Prediction

This notebook performs data acquisition, EDA, preprocessing, model training, evaluation, hyperparameter tuning, MLflow tracking and model packaging for the Heart Disease UCI dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
import joblib
import mlflow
import mlflow.sklearn

sns.set_theme(style="whitegrid")

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
Path("../data/raw").mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_URL, header=None, names=columns, na_values="?")
df["target"] = (df["target"] > 0).astype(int)
df.to_csv("../data/raw/heart_disease_uci.csv", index=False)
df.head()

## Dataset Overview and Missing Value Analysis

In [ ]:
print(df.shape)
display(df.describe())
display(df.isna().sum().to_frame("missing_count"))

## Class Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="target")
plt.title("Class Distribution: 0 = No Disease, 1 = Disease")
plt.show()

## Histograms

In [ ]:
df.hist(figsize=(14,10), bins=20)
plt.suptitle("Feature Histograms", y=1.02)
plt.show()

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## Feature Relationship Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="target", y="thalach")
plt.title("Maximum Heart Rate by Target Class")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="target", y="oldpeak")
plt.title("ST Depression by Target Class")
plt.show()

## Preprocessing Pipeline

In [ ]:
target = "target"
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

X = df[numeric_features + categorical_features]
y = df[target]

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Model Training, Hyperparameter Tuning and MLflow Tracking

In [ ]:
mlflow.set_experiment("heart-disease-uci-mlops")
models = {
    "logistic_regression": (LogisticRegression(max_iter=1000), {"model__C": [0.1, 1, 10], "model__solver": ["liblinear"]}),
    "random_forest": (RandomForestClassifier(random_state=42), {"model__n_estimators": [100, 200], "model__max_depth": [3, 5, None]})
}

results = []
best_model = None
best_auc = -1
best_name = None

for name, (estimator, params) in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    grid = GridSearchCV(pipe, params, scoring="roc_auc", cv=5, n_jobs=-1)
    with mlflow.start_run(run_name=name):
        grid.fit(X_train, y_train)
        preds = grid.predict(X_test)
        probs = grid.predict_proba(X_test)[:, 1]
        metrics = {
            "accuracy": accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds),
            "f1": f1_score(y_test, preds),
            "roc_auc": roc_auc_score(y_test, probs),
            "cv_roc_auc": cross_val_score(grid.best_estimator_, X, y, cv=5, scoring="roc_auc").mean()
        }
        mlflow.log_params(grid.best_params_)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(grid.best_estimator_, "model")
        results.append({"model": name, "best_params": grid.best_params_, **metrics})
        if metrics["roc_auc"] > best_auc:
            best_auc = metrics["roc_auc"]
            best_model = grid.best_estimator_
            best_name = name
pd.DataFrame(results)

## Final Model Evaluation Plots

In [ ]:
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"ROC Curve - {best_name}")
plt.show()

## Save Reusable Pipeline

In [ ]:
Path("../models").mkdir(exist_ok=True)
joblib.dump(best_model, "../models/heart_pipeline.joblib")
print("Saved model to ../models/heart_pipeline.joblib")